# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing entities by their Croissant `@id` fields for reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. 
We'll start by pointing to the Croissant schema URL and inspecting the dataset metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Version: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
We can access the `recordSet` entities in the Croissant metadata. All data tables are defined as `recordSet` objects, and fields/columns within them are referenced by `@id`.

Let's print the available record sets (tables) and their fields by `@id` and label.

In [ ]:
# Get all available record set objects
record_sets = list(metadata.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  - Name: {rs.name}")
    print(f"  - Description: {rs.description}")
    print(f"  - Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id:45} Label: {field.name} | Type: {field.data_type}")
    print()
    record_set_ids.append(rs.id)

# For demonstration, show records from the first record set (using @id)
first_record_set_id = record_set_ids[0]
print(f"\nFirst few records from record set: {first_record_set_id}\n")
for i, x in enumerate(dataset.records(record_set=first_record_set_id)):
    pprint.pprint(x)
    if i >= 2:
        break

## 3. Data Extraction
We'll extract all record sets into Pandas DataFrames, referencing each set by its `@id` as per the Croissant schema. You can use the list of IDs displayed above to select the data you wish to process.

In [ ]:
# Load each record set into a DataFrame, mapping by @id
dataframes = {}

for rs in record_sets:
    recs = list(dataset.records(record_set=rs.id))
    if len(recs) > 0:
        dataframes[rs.id] = pd.DataFrame(recs)
        print(f"Loaded {len(recs)} records for RecordSet @id: {rs.id}")
    else:
        print(f"No records in RecordSet @id: {rs.id}")

# If there are multiple record sets, print their columns
for rsid, df in dataframes.items():
    print(f"\nColumns in DataFrame for RecordSet @id: {rsid}")
    print(df.columns.tolist())
    print(df.head())

# For subsequent analysis, select the primary data record set (usually containing protein measurements)
primary_record_set_id = next(iter(dataframes.keys()))
df = dataframes[primary_record_set_id]  # Alias for convenience


## 4. Exploratory Data Analysis (EDA)
Let's demonstrate typical steps: filtering by a numeric field (such as 'MW' for molecular weight), normalizing it, and grouping by a categorical attribute (`@id`, e.g., protein type or label).

We must select a numeric field and a group field. You can choose other field `@id`s from the printout above as needed.

In [ ]:
# Identify example fields by inspecting the DataFrame columns
print(f"Columns available for EDA in '{primary_record_set_id}':\n", df.columns.tolist())

# Let's pick a numeric field (replace with correct @id from dataset overview, e.g., 'MW' or 'MolecularWeight')
# You may need to adjust these IDs to match the ones found in your printout
numeric_field_id = None
for col in df.columns:
    if 'mw' in col.lower() or 'mass' in col.lower() or 'weight' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback - use the first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
print(f"Using numeric field: {numeric_field_id}")

# For grouping, try a categorical field, such as 'description' or similar
group_field_id = None
for col in df.columns:
    if 'description' in col.lower() or 'type' in col.lower() or 'label' in col.lower():
        group_field_id = col
        break
if group_field_id is None:
    group_field_id = df.columns[0]
print(f"Using group/categorical field: {group_field_id}")

# Filter records: For demonstration, keep records where the numeric field is present and above a threshold
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
else:
    # Some columns may have numeric content as string, try conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field (standard score z=(x-mean)/std)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group records by categorical field and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualizing data: Let's look at a histogram of the selected numeric field, and a boxplot grouped by the chosen categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
filtered_df[numeric_field_id].hist(bins=30)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f'Histogram of {numeric_field_id}')
plt.show()

# Boxplot grouped by group_field_id
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'{numeric_field_id} grouped by {group_field_id}')
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated:
- How to load a Croissant FAIR dataset using `mlcroissant` and reference all entities by `@id`.
- How to enumerate and extract tables, fields, and records from the metadata.
- How to filter, normalize, group, and visualize the data using standard data science tools.

This lightweight EDA provides a starting point for further proteomics or domain-specific analysis on the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa).

For advanced use (such as joining additional metadata tables, record linkage, or more complex processing), always refer to entities by their `@id` field as shown, ensuring reproducibility and clear mapping to the Croissant schema.